In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

results_df = spark.table(
    f"{catalog}.f1_silver.race_results"
)

pit_performance_df = spark.table(
    f"{catalog}.f1_gold.pit_stop_performance"
)

degradation_df = spark.table(
    f"{catalog}.f1_gold.tyre_degradation"
)

In [0]:
driver_results_kpi_df = (
    results_df
    .groupBy(
        "driver_id",
        "driver_code",
        "driver_last_name",
        "constructor_name"
    )
    .agg(
        F.sum("points").alias("total_points"),
        F.avg("finish_position").alias("average_finish_position"),
        F.countDistinct("race_id").alias("races_completed")
    )
)

In [0]:
driver_pit_kpi_df = (
    pit_performance_df
    .groupBy("driver_id")
    .agg(
        F.sum("pit_stop_count").alias("total_pit_stops"),
        F.avg("average_stop_duration_ms").alias(
            "average_pit_stop_duration_ms"
        )
    )
)

In [0]:
driver_degradation_kpi_df = (
    degradation_df
    .groupBy("driver_id")
    .agg(
        F.avg("degradation_ms_per_lap").alias(
            "average_degradation_ms_per_lap"
        ),
        F.count("*").alias("measured_stints")
    )
)

In [0]:
team_driver_kpis_df = (
    driver_results_kpi_df
    .join(
        driver_pit_kpi_df,
        on="driver_id",
        how="left"
    )
    .join(
        driver_degradation_kpi_df,
        on="driver_id",
        how="left"
    )
    .withColumn(
        "average_pit_stop_duration_seconds",
        F.col("average_pit_stop_duration_ms") / 1000
    )
    .select(
        "driver_id",
        "driver_code",
        "driver_last_name",
        "constructor_name",
        "total_points",
        "average_finish_position",
        "races_completed",
        "total_pit_stops",
        "average_pit_stop_duration_seconds",
        "average_degradation_ms_per_lap",
        "measured_stints"
    )
)

display(
    team_driver_kpis_df
    .orderBy(F.desc("total_points"))
)

In [0]:
kpi_table = f"{catalog}.f1_gold.team_driver_kpis"

(
    team_driver_kpis_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(kpi_table)
)

print(f"Saved: {kpi_table}")